**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Optimization

Every 'fit', 'train', and 'design' in this curriculum is secretly `argmin`. Four sessions on why gradient methods work, when they're guaranteed to, what constraints do, and why stochastic noise is a feature — the theory under [ANN's](../../Intro_Mach_Learn/Intro_ANN/Intro_ANN.ipynb) backprop loop and [Adaptive Filtering's](../../Intro_Time_Series/Intro_AdFilt_APA.ipynb) LMS.

## 0. Introduction

The object of study: $\min_x f(x)$. Three questions organize the course — *when is the minimum unique?* (convexity), *how fast do we get there?* (rates), *what if $x$ is constrained?* (Lagrange).

## 1. Pre-requisites

[Linear Algebra](../Linear_Algebra/Linear_Algebra.ipynb) — especially S3 (eigenvalues) and S5 (matrix calculus). Multivariable calculus at the chain-rule level.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 4 — *Convexity* (~35 min)
**Goal:** recognize the functions where local = global; test convexity via the Hessian.
**Builds on:** [Linear Algebra](../Linear_Algebra/Linear_Algebra.ipynb). &nbsp; **Feeds into:** Session 2 (gradient descent).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Convexity</b></summary>

**Timing (~35 min).** 8 min the bowl picture · 10 min the local⇒global proof · 8 min the Hessian test · 9 min the zoo and the honest caveat.

**Board first — draw the chord.** A convex function is one where the straight line between any two points on the graph sits *above* the graph. That single picture is the definition, and the inequality $f(\theta x + (1-\theta)y) \le \theta f(x) + (1-\theta)f(y)$ is just that sentence in symbols. Draw the chord before writing the inequality; students who see the picture never misremember the direction.

**Then the consequence that justifies the whole session.** For a convex function, **any local minimum is the global minimum**. A downhill walker cannot get trapped. That is why convex problems are the ones optimisation can genuinely *promise* to solve, and why the field is organised around the convex/non-convex boundary.

**Do the proof — it is four lines and unusually satisfying.** Suppose $x^*$ is a local min and some $y$ has $f(y) < f(x^*)$. Points on the segment $\theta y + (1-\theta)x^*$ approach $x^*$ as $\theta \to 0$, and convexity bounds their value below $f(x^*)$. So there are points arbitrarily close to $x^*$ with strictly smaller value, contradicting local minimality. Ask which step used convexity; exactly one does, and it is load-bearing.

**The Hessian test connects back to linear algebra.** $\nabla^2 f \succeq 0$ means all eigenvalues non-negative — the bowl curves upward along every direction. For a quadratic $\frac12 x^\top Sx - b^\top x$ the Hessian *is* $S$, so convexity is a statement about $S$'s spectrum. That is [Linear Algebra S3](../Linear_Algebra/Linear_Algebra.ipynb) doing the work, and it previews Session 2 where the *same* eigenvalues set the convergence rate.

**Work the zoo as recognition practice, not a list.** Least squares ($A^\top A \succeq 0$), cross-entropy of a linear model, every norm, and the max of convex functions. Ask the room *why* every norm is convex — the triangle inequality plus homogeneity is exactly the chord condition — rather than accepting it as a fact to memorise.

**Then the honest caveat, stated plainly rather than buried.** Neural network losses are **not** convex, so nothing proved in this session applies to them directly. Everything deep learning does with gradients is running on intuition borrowed from the convex case, and it works better than the theory can currently explain. Say that openly; it makes Session 4's SGD material land as engineering rather than as theory, and it is the honest position.
</details>

## 2. Convex Functions

💡 **Intuition.** A convex function is a **bowl**: the chord between any two points sits above the graph. The consequence worth the whole session: *any local minimum is the global minimum* — a downhill walker cannot get trapped. Convex problems are the ones optimization can truly promise to solve; everything else (deep learning included) lives on borrowed intuition from this case.

**Definition.** $f$ is convex if for all $x, y$ and $\theta \in [0,1]$:
$$f(\theta x + (1-\theta) y) \le \theta f(x) + (1-\theta) f(y)$$

**Second-order test.** Twice-differentiable $f$ is convex iff its Hessian $\nabla^2 f \succeq 0$ (all eigenvalues $\ge 0$) everywhere — the bowl curves up along every axis. For a quadratic $f = \tfrac12 x^T S x - b^T x$, the Hessian *is* $S$: convex iff $S \succeq 0$, strictly (unique minimum) iff $S \succ 0$.

**Proof that local ⇒ global (convex case).** Let $x^\*$ be a local min and suppose $f(y) < f(x^\*)$ for some $y$. Points $\theta y + (1-\theta)x^\*$ approach $x^\*$ as $\theta \to 0$, and convexity gives $f(\theta y + (1-\theta)x^\*) \le \theta f(y) + (1-\theta) f(x^\*) < f(x^\*)$ — arbitrarily close points with lower value, contradicting local minimality. $\blacksquare$

In [2]:
# Convex vs nonconvex, and what a walker experiences
x = np.linspace(-3, 3, 400)
fig, axes = plt.subplots(1, 2, figsize=(9, 2.6))
axes[0].plot(x, x**2 + 0.5 * x); axes[0].set_title("convex: one basin, no traps")
axes[1].plot(x, x**4 - 3 * x**2 + x); axes[1].set_title("nonconvex: local minima exist")
for ax in axes: ax.grid(True)
plt.tight_layout(); plt.show()

/tmp/ipykernel_2012565/3997848261.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**Convexity zoo you already use:** least squares ($A^TA \succeq 0$), cross-entropy of a linear model, every norm ($\|\cdot\|_1, \|\cdot\|_2$), max of convex functions. **Not convex:** neural network losses — yet Session 4 explains why training works anyway.

---
### 🕐 Session 2 of 4 — *Gradient Descent & Rates* (~40 min)
**Goal:** prove how fast GD converges on smooth/strongly-convex problems; see conditioning bite.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (constraints).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Gradient Descent & Rates</b></summary>

**Timing (~40 min).** 10 min $L$ and $\mu$ as trust and floor · 10 min the rate theorem · 12 min the two demos · 8 min what attacks $\kappa$.

**Board first — give $L$ and $\mu$ physical readings.** $L$ bounds curvature from above: *how far can I trust the slope?* A step of about $1/L$ is safe, and larger overshoots. $\mu$ bounds it from below: *does the bowl ever flatten out?* If it does, progress stalls near the bottom. Their ratio $\kappa = L/\mu$ is the **elongation of the bowl**, and it is the villain of the entire course.

**Make the rate concrete before showing it.** Error shrinks by $(1 - 1/\kappa)$ per step. Have the room compute how many steps that needs for a $10^{-6}$ reduction at $\kappa = 25$: about 338. Then at $\kappa = 10^4$: about 138,000. **The same algorithm, the same code, a 400× difference in cost — decided entirely by the problem's geometry.** That is the moment $\kappa$ stops being a symbol.

**Use the two-panel demo as a picture of what $\kappa$ *feels* like.** At $\kappa = 2$ the contours are nearly circular and the path runs straight to the bottom. At $\kappa = 25$ they are a canyon and the path zigzags across it, making slow progress along its length. Point out *why*: the gradient points perpendicular to the contours, which in a canyon means mostly across rather than along.

**The measured-rate cell is a prediction, so treat it as one.** Have the room compute $1 - \mu/L = 0.96$ from the eigenvalues *before* running, then reveal the measured 0.9600. Predicting a dynamical rate from a spectrum and hitting it to four decimals is the satisfying moment of the session.

**Then the payoff: everything in the optimiser zoo attacks $\kappa$.** Preconditioning changes coordinates to round the bowl. Momentum and CG extract $\sqrt\kappa$ instead of $\kappa$ ([Numerical Linear Algebra](../Numerical_Linear_Algebra/Numerical_Linear_Algebra.ipynb) S3). Adam approximates a per-coordinate rescaling. Newton's method uses the exact Hessian and gets $\kappa = 1$ at cubic cost. Framing them as four answers to *one* question is far more useful than four separate recipes.

**And the cross-reference worth making.** The step-size bound $\eta < 2/L$ here is the same condition as $\mu < 2/\lambda_{\max}$ in [LMS](../../Intro_Time_Series/Intro_AdFilt_APA.ipynb), and the same $\kappa$ governs [RLS](../../Intro_Time_Series/Intro_RLS.ipynb)'s advantage over gradient methods and the digit loss in [Numerical Linear Algebra](../Numerical_Linear_Algebra/Numerical_Linear_Algebra.ipynb) S1. One number, four workshops.
</details>

## 3. Gradient Descent

💡 **Intuition.** GD's guarantee needs two numbers: $L$ (curvature never exceeds $L$ — you can trust the slope for a step of about $1/L$) and $\mu$ (curvature never below $\mu$ — the bowl never flattens out). Their ratio $\kappa = L/\mu$, the **condition number**, is the villain of the course: it is the elongation of the bowl, and error shrinks by a factor $\approx (1 - 1/\kappa)$ per step. Round bowl ⇒ sprint; canyon ⇒ zigzag crawl. For quadratics, $L$ and $\mu$ are just the extreme eigenvalues from [Linear Algebra S3](../Linear_Algebra/Linear_Algebra.ipynb).

**Theorem (rate, stated).** For $L$-smooth, $\mu$-strongly-convex $f$, GD with step $\eta = 1/L$ satisfies
$$\|x_k - x^*\|^2 \le \big(1 - \tfrac{\mu}{L}\big)^k \, \|x_0 - x^*\|^2$$
— *linear convergence*: a fixed fraction of the remaining error removed per step. (Proof for quadratics is a two-line eigen-argument: the error multiplies by $I - \eta S$, whose eigenvalues are $1 - \eta \lambda_i$.)

In [3]:
# Watch κ control the speed, exactly as the theorem says
def gd_path(S, x0, eta, steps=60):
    xs = [np.array(x0, float)]
    for _ in range(steps):
        xs.append(xs[-1] - eta * S @ xs[-1])
    return np.array(xs)

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.4))
for ax, (l1, l2) in zip(axes, [(1.0, 2.0), (1.0, 25.0)]):
    S = np.diag([l1, l2])
    path = gd_path(S, [2.6, 1.8], eta=1/l2)
    g = np.linspace(-3, 3, 100)
    GX, GY = np.meshgrid(g, g)
    ax.contour(GX, GY, l1*GX**2/2 + l2*GY**2/2, levels=12, alpha=0.5)
    ax.plot(*path.T, "o-", markersize=2.5, color="crimson")
    kappa = l2 / l1
    ax.set_title(f"κ = {kappa:.0f}: {'sprints' if kappa < 5 else 'zigzags'}")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2012565/913284460.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** The same algorithm, the same code, two problems — and completely different behaviour. At $\kappa = 2$ the contours are nearly circular and the path runs almost straight to the bottom. At $\kappa = 25$ they form a canyon and the path **zigzags** across it, creeping along its length.

**The zigzag has a precise cause.** The gradient points perpendicular to the contour lines. In a circular bowl that is straight at the minimum. In an elongated canyon it points mostly *across* the valley rather than *along* it — so most of each step is spent bouncing between the walls, and only a small component makes real progress toward the bottom. Steepest descent is steepest *locally*, and locally steepest is not the direction you want.

**Note also that the step size is capped by the wrong eigenvalue.** With $\eta = 1/L$ set by the *largest* curvature, the step is chosen to be safe in the steep direction — and that same step is far too small for the shallow direction, where progress needs to happen. Both symptoms come from one number.

**Convert $\kappa$ into a cost, because that is what makes it real.** Error shrinks by $(1 - 1/\kappa)$ per step, so reducing it by $10^{-6}$ takes roughly:

| $\kappa$ | iterations |
|---|---|
| 2 | ~20 |
| 25 | ~338 |
| $10^4$ | ~138,000 |

Identical code, a 400× difference in cost between the last two rows, decided entirely by the **geometry of the problem** rather than by anything about the algorithm. That is why $\kappa$ is the villain of this workshop.

**And it makes the optimiser zoo comprehensible as one idea.** Every method you have heard of attacks $\kappa$:

- **Preconditioning** changes coordinates to round the bowl out.
- **Momentum** and **conjugate gradients** extract $\sqrt\kappa$ instead of $\kappa$ — at $\kappa = 10^4$ that is 100× fewer iterations ([Numerical Linear Algebra](../Numerical_Linear_Algebra/Numerical_Linear_Algebra.ipynb) S3).
- **Adam** approximates a per-coordinate rescaling, which is a cheap diagonal preconditioner.
- **Newton's method** uses the exact Hessian and achieves $\kappa = 1$, at cubic cost per step.

Four techniques, one motivation. Students who see them as answers to a single question retain far more than those who meet them as four unrelated recipes.

**And the same number appears throughout this curriculum.** It is [LMS](../../Intro_Time_Series/Intro_AdFilt_APA.ipynb)'s convergence penalty on correlated input, [RLS](../../Intro_Time_Series/Intro_RLS.ipynb)'s reason for carrying $R^{-1}$, and the digit-loss factor in [Numerical Linear Algebra](../Numerical_Linear_Algebra/Numerical_Linear_Algebra.ipynb). One eigenvalue ratio, four workshops.

In [4]:
# Measured rate vs predicted (1 − μ/L) per step
l1, l2 = 1.0, 25.0
path = gd_path(np.diag([l1, l2]), [2.6, 1.8], eta=1/l2, steps=200)
err = np.linalg.norm(path, axis=1)
measured = (err[-1] / err[100]) ** (1 / 100)
print(f"measured per-step factor {measured:.4f}   theory 1 − μ/L = {1 - l1/l2:.4f}")

measured per-step factor 0.9600   theory 1 − μ/L = 0.9600


This is *why* preconditioning, momentum, and Adam exist: they all attack $\kappa$. And it's the same $\lambda_{max}$ speed limit you met as $\mu < 2/\lambda_{max}$ in [LMS](../../Intro_Time_Series/Intro_AdFilt_APA.ipynb).

---
### 🕐 Session 3 of 4 — *Constraints: Lagrange & KKT* (~35 min)
**Goal:** optimize with equality and inequality constraints; read a Lagrangian like a force balance.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (SGD).

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: Constraints — Lagrange & KKT</b></summary>

**Timing (~35 min).** 10 min the force-balance picture · 8 min Lagrange · 8 min KKT's two extra conditions · 9 min the eigenvalue payoff.

**Board first — the geometric statement, before any algebra.** At a constrained optimum you cannot improve *without leaving the feasible set*. So the objective's downhill direction must be exactly opposed by the constraint surface — $\nabla f$ is parallel to $\nabla g$. Draw contours of $f$ and the constraint curve, and show that at the optimum they are **tangent**: if they crossed, you could slide along the constraint and improve. That picture *is* the Lagrange condition, and the multiplier is just the proportionality constant.

**Give $\lambda$ two readings and use both.** Physically it is the strength of the wall force. Economically it is the **shadow price** — how much the optimum improves per unit of constraint relaxation. The second reading is what makes multipliers useful in practice, and it is the same interpretation that appears in [Convex Optimization II](./Convex_Optimization_2.ipynb)'s duality session.

**KKT's two additions each have a one-line justification.** $\lambda \ge 0$: walls only *push*, never pull — an inequality constraint cannot drag you toward it. Complementary slackness $\lambda h = 0$: a constraint you are not touching exerts no force, so either it is active or its multiplier is zero. Ask the room to state each in words before writing the symbols; both are obvious once phrased physically and opaque as algebra.

**The eigenvalue example is the session's payoff — do not rush it.** Minimising $x^\top Sx$ subject to $\|x\| = 1$ gives stationarity $Sx = \lambda x$. So the constrained minimiser is an **eigenvector**, and the optimal value is its eigenvalue. Let that land: *eigenproblems are constrained optimisation problems*. Students have computed eigenvectors since [Linear Algebra](../Linear_Algebra/Linear_Algebra.ipynb) without knowing what they optimise.

**Then name where that shows up, because it is everywhere.** PCA maximises variance on the unit sphere. MVDR beamforming in [Array Processing](../../Intro_DSP/Array_Processing.ipynb) minimises output power subject to unit gain — the same shape, with the answer $R^{-1}a/(a^HR^{-1}a)$ coming straight out of the Lagrangian. [Manifold Optimization](./Manifold_Optimization.ipynb) is what happens when you solve the same problem by *walking on the constraint* instead of using multipliers.

**On the demo's small mismatch.** The numerical minimiser is $[-0.9864, 0.1643]$ against the eigenvector's $[-0.9863, 0.1648]$ — a difference of about $5\times10^{-4}$. That is *grid resolution*, not error: 2000 samples around the circle gives an angular spacing of $3.1\times10^{-3}$ radians, so components can be off by roughly half that. The objective values agree exactly (0.1221 both), which is the expected pattern near a minimum — the function is flat there, so the *value* is far better determined than the *location*.
</details>

## 4. Constrained Optimization

💡 **Intuition.** At a constrained optimum you cannot improve *without leaving the feasible set* — so the objective's downhill direction must be exactly opposed by the constraint's 'wall'. Algebraically: $\nabla f$ is a combination of constraint normals. The multipliers $\lambda$ are the strengths of the wall forces — and economically, the *price* of tightening each constraint.

**Lagrange (equality).** For $\min f$ s.t. $g(x) = 0$: at a (regular) optimum $\exists \lambda$ with $\nabla f = \lambda \nabla g$. Solve by stationarity of $\mathcal{L}(x, \lambda) = f(x) - \lambda g(x)$.

**KKT (inequality, $h(x) \le 0$)** adds two conditions: $\lambda \ge 0$ (walls only push, never pull) and *complementary slackness* $\lambda h(x) = 0$ (an inactive constraint exerts no force).

**Worked example.** $\min\; x^T S x$ s.t. $\|x\| = 1$: stationarity gives $S x = \lambda x$ — the constrained minimizer is the **smallest eigenvector**. Eigenproblems *are* constrained optimization; this is how PCA, MVDR beamforming ([Array Processing](../../Intro_DSP/Array_Processing.ipynb)), and Rayleigh quotients arise.

In [5]:
# Verify: minimize xᵀSx on the unit circle — numerically vs the eigen-answer
M = rng.standard_normal((2, 2)); S = M @ M.T + 0.1 * np.eye(2)
th = np.linspace(0, 2 * np.pi, 2000)
circle = np.stack([np.cos(th), np.sin(th)])
vals = np.einsum("ij,ji->i", circle.T @ S, circle)
x_num = circle[:, vals.argmin()]

w, V = np.linalg.eigh(S)
print("numerical minimizer:", np.round(x_num, 4), " value", vals.min().round(4))
print("smallest eigenvector:", np.round(V[:, 0], 4), " eigenvalue", w[0].round(4))

numerical minimizer: [-0.9864  0.1643]  value 0.1221
smallest eigenvector: [-0.9863  0.1648]  eigenvalue 0.1221


**What just happened.** A brute-force search over 2000 points on the unit circle found the minimiser $[-0.9864, 0.1643]$ with value $0.1221$. `eigh` returned the smallest eigenvector $[-0.9863, 0.1648]$ with eigenvalue $0.1221$. **Same point, same value — one found by searching, one by an eigen-decomposition.**

**The Lagrangian explains why they must agree.** Minimising $x^\top Sx$ subject to $\|x\|^2 = 1$ gives stationarity
$$2Sx = 2\lambda x \quad\Longrightarrow\quad Sx = \lambda x,$$
which is the eigenvalue equation. So the constrained minimiser *is* an eigenvector, and the multiplier $\lambda$ *is* the eigenvalue — which is also the optimal objective value, since $x^\top Sx = \lambda x^\top x = \lambda$ on the unit sphere.

**Let that land, because it reframes a familiar object.** Eigenproblems **are** constrained optimisation problems. Students have been computing eigenvectors since [Linear Algebra](../Linear_Algebra/Linear_Algebra.ipynb) without a statement of what they optimise; the answer is "the quadratic form, on the unit sphere," with the smallest eigenvector minimising and the largest maximising.

That single fact explains a great deal of this curriculum at once. **PCA** maximises variance subject to unit norm. **MVDR beamforming** in [Array Processing](../../Intro_DSP/Array_Processing.ipynb) minimises output power subject to unit gain in the look direction, and its solution $R^{-1}a/(a^HR^{-1}a)$ drops straight out of the Lagrangian. The **Rayleigh quotient** in [Manifold Optimization](./Manifold_Optimization.ipynb) is the same problem solved by walking on the sphere instead of using a multiplier. Four workshops, one Lagrangian.

**On the small mismatch in the fourth decimal.** The two vectors differ by about $5\times10^{-4}$, and that is **grid resolution rather than error**: 2000 samples around the circle gives an angular spacing of $3.1\times10^{-3}$ radians, so the brute-force answer can be off by roughly half a grid step. Notice that the objective *values* agree exactly (0.1221 both) while the *locations* differ slightly — the characteristic pattern near any minimum, where the function is flat and so the value is far better determined than the argument. Worth remembering when you evaluate an optimiser: matching objective values is weaker evidence than matching arguments.

**And the multiplier has a reading beyond the algebra.** $\lambda$ measures how much the objective would improve per unit of constraint relaxation — the **shadow price** of the constraint. Physically, it is the strength of the wall force that stops the downhill walker from leaving the feasible set. That interpretation is what makes multipliers practically useful, and it is the same object that becomes the dual variable in [Convex Optimization II](./Convex_Optimization_2.ipynb).

---
### 🕐 Session 4 of 4 — *Stochastic Gradient Descent* (~40 min)
**Goal:** understand SGD's noise: why it's cheap, why it still converges, and why it can even help.
**Builds on:** Sessions 2–3.

---

<details>
<summary>🎓 <b>Teacher notes — Session 4: Stochastic Gradient Descent</b></summary>

**Timing (~40 min).** 8 min the cost argument · 10 min unbiasedness and why it still works · 12 min the noise-floor demo · 10 min decay schedules and the nonconvex bonus.

**Board first — the economics, because that is the actual motivation.** A full gradient costs a pass over *all* the data. On a million examples that is a million evaluations for **one** step. SGD uses a mini-batch of 8 and takes 125,000 steps for the same cost. Ask which makes more progress; the answer is obviously the second, provided the noisy steps are not systematically wrong.

**Then the property that makes it legitimate.** The mini-batch gradient is **unbiased**: right on average. So each step is downhill *in expectation*, and averaging over many steps mimics averaging over data — the LLN from [Independence](../Analysis/Independence.ipynb) doing the work. SGD is not an approximation that happens to work; it is an unbiased estimator plugged into a method that tolerates unbiased noise.

**The two-term decomposition is the session's organising equation.** $E\|x_k - x^*\|^2 \lesssim (1-\eta\mu)^k\|x_0-x^*\|^2 + \eta\sigma^2/\mu$: a **bias** term that shrinks geometrically and a **noise floor** that does not. Read each factor aloud — the floor is proportional to $\eta$, so halving the step size halves the floor and slows the approach. That is the entire trade, in one line.

**Have the room predict the demo before running it.** Three curves: $\eta = 0.01$ (fast, high floor), $\eta = 0.001$ (slow, low floor), decaying (both). Ask which they would choose for a fixed compute budget — and the answer genuinely depends on the budget, which is the point. Then reveal that decay gets both, because it is fast early when far away and small late when settling.

**Name the practice this justifies.** Learning-rate schedules in deep learning — step decay, cosine annealing, warmup-then-decay — are all this equation. A model that has plateaued and then improves sharply when the LR drops is *exactly* the floor $\eta\sigma^2/\mu$ moving down. Students who have seen that in a training curve without explanation find this satisfying.

**Then the nonconvex bonus, stated carefully.** In a nonconvex landscape the noise *helps*: it rattles the iterate out of narrow, sharp minima while broad flat ones retain it. Broad minima empirically generalise better, so SGD's noise acts as an implicit regulariser. Be honest that this is well-supported empirically and not fully explained theoretically — the mechanism is plausible and the flat-minima/generalisation link is still debated. It is the honest frontier, and [Training Dynamics](../../Intro_Mach_Learn/Training_Dynamics.ipynb) picks it up.

**Close on the arc.** Convexity was the promise, $\kappa$ the speed limit, Lagrange the wall forces, and SGD the affordable gamble. Every knob in a modern training script is an intervention on one of those four.
</details>

## 5. SGD

💡 **Intuition.** Full gradients cost a pass over ALL data; SGD gambles on a mini-batch's estimate. The estimate is **unbiased** — right on average — so each step is downhill *in expectation*, and averaging over steps mimics averaging over data (the LLN from [Independence](../Analysis/Independence.ipynb)). The price: a noise floor set by step size × gradient variance. The classic cure: **decay the step size** — big steps to travel, small steps to settle. And in nonconvex landscapes the noise moonlights as an explorer, rattling the iterate out of narrow bad minima.

**The trade in one equation** (strongly convex case, stated): with constant step $\eta$,
$$E\|x_k - x^*\|^2 \lesssim \underbrace{(1 - \eta\mu)^k \|x_0 - x^*\|^2}_{\text{bias: shrinks}} + \underbrace{\frac{\eta \, \sigma^2}{\mu}}_{\text{noise floor: doesn't}}$$
Decaying $\eta_k \propto 1/k$ drives both terms to zero (at the slower $O(1/k)$ rate).

In [6]:
# See the noise floor and the decay cure, on least squares
A = rng.standard_normal((2000, 20)); x_true = rng.standard_normal(20)
b = A @ x_true + 0.5 * rng.standard_normal(2000)
x_star, *_ = np.linalg.lstsq(A, b, rcond=None)

def sgd(eta_fn, steps=8000, batch=8):
    x = np.zeros(20); errs = []
    for k in range(steps):
        i = rng.integers(0, len(A), batch)
        g = 2 * A[i].T @ (A[i] @ x - b[i]) / batch
        x -= eta_fn(k) * g
        if k % 20 == 0: errs.append(np.linalg.norm(x - x_star))
    return np.array(errs)

plt.figure(figsize=(8, 3))
for label, fn in [("η = 0.01 (floor!)", lambda k: 0.01),
                  ("η = 0.001 (lower floor, slower)", lambda k: 0.001),
                  ("η = 0.01/(1+k/1000) (decay: best of both)", lambda k: 0.01 / (1 + k / 1000))]:
    plt.semilogy(np.arange(0, 8000, 20), sgd(fn), label=label, alpha=0.8)
plt.legend(); plt.grid(True, alpha=0.3)
plt.xlabel("iteration"); plt.ylabel("‖x − x*‖")
plt.title("SGD: converge fast OR settle low — decay schedules buy both")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2012565/4068096982.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Three step-size strategies, and the plot shows exactly the trade the theory predicts:

- **$\eta = 0.01$** — descends fast, then **flattens** at a visible floor and stops improving no matter how long it runs.
- **$\eta = 0.001$** — descends more slowly, and settles at a *lower* floor.
- **Decaying $\eta$** — matches the fast curve early and the low curve late.

**The floor is not a bug and it does not go away with patience.** The governing decomposition is
$$E\|x_k - x^*\|^2 \;\lesssim\; \underbrace{(1-\eta\mu)^k\|x_0-x^*\|^2}_{\text{bias — shrinks geometrically}} \;+\; \underbrace{\frac{\eta\sigma^2}{\mu}}_{\text{noise floor — constant}}.$$
The first term vanishes; the second does not depend on $k$ at all. So a constant step size buys you geometric progress *until* you reach a floor proportional to $\eta$, and then nothing. Halving $\eta$ halves the floor and halves the speed — which is precisely the gap between the first two curves.

**Which makes the decay schedule obvious rather than clever.** Large steps while far away (where speed matters and precision does not), small steps once close (where the floor matters and speed does not). $\eta_k \propto 1/(1+k/1000)$ drives both terms to zero, at the slower $O(1/k)$ rate — you give up geometric convergence in exchange for actually arriving.

**And this is the theory behind every learning-rate schedule you have seen.** Step decay, cosine annealing, warmup-then-decay: all of them are this equation. If you have ever watched a training loss plateau and then drop sharply the moment the learning rate was reduced, you were watching $\eta\sigma^2/\mu$ move down. It is not the model suddenly learning something new; it is the noise floor being lowered.

**Why SGD is worth the noise at all.** A full gradient costs a pass over *all* the data — on a million examples, a million evaluations for **one** step. A batch of 8 gives 125,000 steps for the same cost. The mini-batch gradient is **unbiased**, so each step is downhill in expectation and the errors average out over many steps ([the LLN](../Analysis/Independence.ipynb) again). SGD is not a crude approximation that happens to work; it is an unbiased estimator inside a method that tolerates unbiased noise.

**One honest note on the nonconvex claim.** The intuition cell says SGD's noise "moonlights as an explorer," rattling iterates out of narrow bad minima. That is well-supported empirically and *not* fully explained theoretically — the flat-minima-generalise-better link remains debated, and none of this session's proofs cover the nonconvex case. It is a plausible mechanism with good evidence, not a theorem, and [Training Dynamics](../../Intro_Mach_Learn/Training_Dynamics.ipynb) takes it up as an open question rather than a settled one.

## 6. Conclusion

Convexity is the promise, $\kappa$ the speed limit, Lagrange the wall forces, and SGD the affordable gamble with a noise floor you now know how to lower. Deep-learning practice ([Training Dynamics](../../Intro_Mach_Learn/Training_Dynamics.ipynb)) is engineering against exactly these quantities.

---
## Where next

- [Training Dynamics](../../Intro_Mach_Learn/Training_Dynamics.ipynb) — Adam, schedules, and regularization as applied versions of these ideas.
- [Adaptive Filtering](../../Intro_Time_Series/Intro_AdFilt_APA.ipynb) — SGD in real time, under the name LMS.
- [Estimation Theory](../Estimation_Theory/Estimation_Theory.ipynb) — what the minimum *means* statistically.